# EcoSort AI: retraining with weighted DataLoaders
Only the **loaders** change versus the previous EfficientNetV2-S feature-extractor run: the same model, loss (plain `CrossEntropyLoss`), optimizer, lr, epochs and batch size. Train and validation loaders now draw samples with probability proportional to `1 / class_size`, so every class is seen about equally often. The dataset is untouched.

Put this notebook next to `data_setup.py`, `engine.py` and `utils.py` (use the updated copies).

In [ ]:
import torch, torchvision
from torch import nn
from pathlib import Path
import data_setup, engine
from utils import save_model, plot_loss_curves

device = 'cuda' if torch.cuda.is_available() else 'cpu'

data_path = Path('data/')
train_dir, val_dir = data_path / 'train', data_path / 'val'

## Weighted train and validation loaders

In [ ]:
weights = torchvision.models.EfficientNet_V2_S_Weights.DEFAULT
auto_transforms = weights.transforms()

train_dataloader, val_dataloader, class_names = data_setup.create_weighted_dataloaders(
    train_dir=train_dir,
    val_dir=val_dir,
    transform=auto_transforms,
    batch_size=32,
    num_workers=0,
    seed=42)
class_names

Sanity check: the labels drawn from the weighted loaders should be roughly uniform across the four classes.

In [ ]:
from collections import Counter

def label_counts(loader, max_batches=100):
    c = Counter()
    for i, (_, y) in enumerate(loader):
        if i >= max_batches: break
        c.update(y.tolist())
    return {class_names[k]: v for k, v in sorted(c.items())}

print('train (first 100 batches):', label_counts(train_dataloader))
print('val   (first 100 batches):', label_counts(val_dataloader))

## Model (identical setup to the previous run)
A fresh EfficientNetV2-S with a frozen feature extractor and a new classifier head, so the only difference is the data feeding.

In [ ]:
torch.manual_seed(42)
torch.cuda.manual_seed(42)

model_2 = torchvision.models.efficientnet_v2_s(weights=weights).to(device)

for param in model_2.features.parameters():
    param.requires_grad = False

model_2.classifier = nn.Sequential(
    nn.Dropout(p=0.2, inplace=True),
    nn.Linear(in_features=1280, out_features=len(class_names))).to(device)

## Train with the existing `engine.train`

In [ ]:
loss_fn = nn.CrossEntropyLoss()   # unweighted: the balancing is done by the loader only
optimizer = torch.optim.Adam(model_2.parameters(), lr=0.01)

from timeit import default_timer as timer
start_time = timer()
results_weighted = engine.train(model=model_2,
                                train_dataloader=train_dataloader,
                                val_dataloader=val_dataloader,
                                loss_fn=loss_fn,
                                optimizer=optimizer,
                                epochs=5,
                                device=torch.device(device))
print(f"[INFO] Total training time: {timer()-start_time:.3f} seconds")

In [ ]:
save_model(model_2, target_dir='models', model_name='03_pretrained_CNN_weighted_model.pt')

## Performance curves
The dashed line is the previous (unweighted) run's validation F1, taken from your earlier notebook output. Note it was measured on the natural validation distribution, while the solid curves here use the balanced validation draw. See the next section for a like-for-like comparison.

In [ ]:
PREV_VAL_F1 = [0.7976, 0.8347, 0.8140, 0.8031, 0.8317]   # previous EfficientNetV2-S run
plot_loss_curves(results_weighted, compare={'previous run val_f1 (natural val)': PREV_VAL_F1})

## Like-for-like check on the natural validation set
The weighted validation loader measures performance on a *balanced* mix. To compare fairly with the earlier run, evaluate the new model on the ordinary, unweighted validation loader. The labels are collected from the loader itself, because a sampler changes the sample order.

In [ ]:
from sklearn.metrics import classification_report
from torchmetrics import ConfusionMatrix
from mlxtend.plotting import plot_confusion_matrix

_, natural_val_dataloader, _ = data_setup.create_dataloaders(
    train_dir, val_dir, auto_transforms, batch_size=32, num_workers=0)

model_2.eval()
preds, targets = [], []
with torch.inference_mode():
    for X, y in natural_val_dataloader:
        logits = model_2(X.to(device))
        preds.append(logits.argmax(dim=1).cpu())
        targets.append(y)
preds, targets = torch.cat(preds), torch.cat(targets)

print(classification_report(targets, preds, target_names=class_names, digits=4))

confmat = ConfusionMatrix(task='multiclass', num_classes=len(class_names))
fig, ax = plot_confusion_matrix(conf_mat=confmat(preds=preds, target=targets).numpy(),
                                class_names=class_names, figsize=(10, 7))